# Real-Time Web Search with Claude Using free-web-search-ultimate v13.0.0

This cookbook demonstrates how to give Claude real-time web search capabilities using the [free-web-search-ultimate](https://github.com/wd041216-bit/free-web-search-ultimate) Python package — **completely free, no API keys required**.

[![free-web-search-ultimate MCP server](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate/badges/score.svg)](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate)

## What you'll learn

- How to use `free-web-search-ultimate` as a Claude tool for real-time web search
- How to build a research assistant that can search the web without any API costs
- How to combine web search results with Claude's reasoning capabilities

## Why free-web-search-ultimate?

| Feature | Details |
|---------|--------|
| **Cost** | Completely free — no API key, no subscription |
| **Privacy** | Uses DuckDuckGo and privacy-respecting search engines |
| **Reliability** | Multiple search backends with automatic fallback |
| **MCP Support** | Works as an MCP server for Claude Desktop and other clients |
| **CLI Support** | Also works as a standalone CLI tool |

## Setup

```bash
pip install free-web-search-ultimate==13.0.0  # v13.0.0 latest anthropic
```

In [ ]:
# Install required packages
%pip install free-web-search-ultimate==13.0.0  # v13.0.0 latest anthropic --quiet

In [ ]:
import anthropic
import os
from free_web_search.search_web import UltimateSearcher

# Initialize the Anthropic client
client = anthropic.Anthropic()

# Initialize the searcher
searcher = UltimateSearcher()


def web_search(
    query: str,
    max_results: int = 5,
    timelimit: str | None = None,
) -> str:
    """Search the web for current information.

    Args:
        query: The search query string.
        max_results: Maximum number of results to return. Defaults to 5.
        timelimit: Optional time filter ('d' for day, 'w' for week,
            'm' for month, 'y' for year).

    Returns:
        A formatted string containing search results with titles, URLs,
        and snippets.
    """
    results = searcher.search(query, max_results=max_results, timelimit=timelimit)
    if not results:
        return "No results found."
    output = []
    for r in results:
        output.append(f"Title: {r.get('title', 'N/A')}")
        output.append(f"URL: {r.get('url', 'N/A')}")
        output.append(f"Snippet: {r.get('body', 'N/A')}")
        output.append("")
    return "\n".join(output)


def news_search(
    query: str,
    max_results: int = 5,
    timelimit: str | None = "w",
) -> str:
    """Search for recent news articles.

    Args:
        query: The search query string.
        max_results: Maximum number of results to return. Defaults to 5.
        timelimit: Time filter for news recency ('d' for day, 'w' for week,
            'm' for month). Defaults to 'w' (past week).

    Returns:
        A formatted string containing news results with titles, URLs,
        and snippets.
    """
    results = searcher.search_news(query, max_results=max_results, timelimit=timelimit)
    if not results:
        return "No news found."
    output = []
    for r in results:
        output.append(f"Title: {r.get('title', 'N/A')}")
        output.append(f"URL: {r.get('url', 'N/A')}")
        output.append(f"Snippet: {r.get('body', 'N/A')}")
        output.append("")
    return "\n".join(output)


## Define the Web Search Tool

You'll define two tools for Claude:
1. `web_search` — General web search
2. `news_search` — Search for recent news

In [ ]:
# Define the tools for Claude
tools = [
    {
        "name": "web_search",
        "description": (
            "Search the web for current information. Use this for general queries, "
            "facts, and topics that may have changed recently."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query",
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results (default: 5)",
                    "default": 5,
                },
                "timelimit": {
                    "type": "string",
                    "description": "Time filter: 'd' (day), 'w' (week), 'm' (month), 'y' (year)",
                    "enum": ["d", "w", "m", "y"],
                },
            },
            "required": ["query"],
        },
    },
    {
        "name": "news_search",
        "description": (
            "Search for recent news articles. Use this for current events, "
            "breaking news, and time-sensitive topics."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The news search query",
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results (default: 5)",
                    "default": 5,
                },
                "timelimit": {
                    "type": "string",
                    "description": "Time filter: 'd' (day), 'w' (week), 'm' (month)",
                    "default": "w",
                    "enum": ["d", "w", "m"],
                },
            },
            "required": ["query"],
        },
    },
]


## Build the Research Assistant

Now let's create a function that lets Claude use web search to answer questions.

In [ ]:
def research_assistant(
    question: str,
    verbose: bool = True,
    max_iterations: int = 10,
) -> str:
    """A research assistant that uses Claude with web search tools.

    Args:
        question: The research question to answer.
        verbose: Whether to print intermediate steps. Defaults to True.
        max_iterations: Maximum number of tool-call iterations to prevent
            infinite loops. Defaults to 10.

    Returns:
        The final answer string from Claude.
    """
    messages = [{"role": "user", "content": question}]

    for iteration in range(max_iterations):
        response = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=4096,
            tools=tools,
            messages=messages,
        )

        if verbose:
            print(f"Iteration {iteration + 1}: stop_reason={response.stop_reason}")

        # If Claude is done, extract and return the final text
        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text
            return ""

        # Process tool calls
        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input
                    if verbose:
                        print(f"  Tool: {tool_name}({tool_input})")

                    if tool_name == "web_search":
                        result = web_search(**tool_input)
                    elif tool_name == "news_search":
                        result = news_search(**tool_input)
                    else:
                        result = f"Unknown tool: {tool_name}"

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })

            # Add assistant response and tool results to messages
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})
        else:
            break

    return f"Reached maximum iterations ({max_iterations}) without a final answer."


## Example 1: Current Events

Let's ask Claude about something that requires up-to-date information.

In [ ]:
# Set to True to run the live search examples below.
# Leave as False to avoid outbound network requests on fresh checkout.
RUN_EXAMPLES = False


In [ ]:
if RUN_EXAMPLES:
    answer = research_assistant("What are the latest developments in AI in 2025?")

## Example 2: News Search

Now let's search for recent news on a specific topic.

In [ ]:
if RUN_EXAMPLES:
    answer = research_assistant("What's the latest news about open-source AI models?")

## Example 3: Research with Multiple Searches

Claude can perform multiple searches to gather comprehensive information.

In [ ]:
if RUN_EXAMPLES:
    answer = research_assistant(
        "Compare the performance of the top 3 open-source LLMs available today. "
        "Include their benchmark scores and key features."
    )

## Using free-web-search as a CLI Tool

The package also works as a standalone CLI tool:

In [ ]:
# You can also use it directly from the command line:
# search-web "latest AI news"
# search-web --type news "AI breakthroughs this week"
# search-web --max-results 10 "Python tutorials"


## Summary

In this cookbook, we've demonstrated how to:

1. **Install** `free-web-search-ultimate` — a free, no-API-key web search library
2. **Define Claude tools** for web search and news search
3. **Build a research assistant** that uses Claude's tool-use capabilities with real-time web search
4. **Handle multi-turn tool use** where Claude performs multiple searches to answer complex questions

### Key Benefits

- **Zero cost**: No API keys or subscriptions needed for web search
- **Privacy-first**: Uses DuckDuckGo and other privacy-respecting search engines
- **Easy integration**: Works with Claude's native tool-use API
- **MCP compatible**: Can also be used as an MCP server for Claude Desktop

### Resources

- [free-web-search-ultimate on GitHub](https://github.com/wd041216-bit/free-web-search-ultimate)
- [free-web-search-ultimate on PyPI](https://pypi.org/project/free-web-search-ultimate/)
- [Glama MCP Server Page](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate)
- [Claude Tool Use Documentation](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)